# TradeGraph (Our Hybrid Model)

## 1 : Importation of the libraries

In [1]:
# ==========================================================
# Cell 1 : Import Libraries
# ==========================================================

import os
import random
import copy
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.nn import GCNConv

from torch_geometric.nn import (
    GATConv,
    global_mean_pool
)

from torch_geometric.nn import SAGEConv

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

import optuna

warnings.filterwarnings("ignore")

# ----------------------------------------------------------
# Reproducibility
# ----------------------------------------------------------

SEED = 82

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

# ----------------------------------------------------------
# Device
# ----------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 70)
print("Environment Information")
print("=" * 70)

print(f"PyTorch version       : {torch.__version__}")

try:
    import torch_geometric
    print(f"PyTorch Geometric     : {torch_geometric.__version__}")
except:
    pass

print(f"Optuna version        : {optuna.__version__}")
print(f"Device                : {device}")

if torch.cuda.is_available():
    print(f"GPU                    : {torch.cuda.get_device_name(0)}")

Environment Information
PyTorch version       : 2.13.0+cu130
PyTorch Geometric     : 2.8.0
Optuna version        : 4.9.0
Device                : cuda
GPU                    : NVIDIA RTX A4500


## 2: Importation of dataset

In [2]:
# ==========================================================
# Cell 2 : Load Cleaned Dataset
# ==========================================================

file_path = "International_Trade_Semiconductor_Cleaned_Data_1995_2024.csv"

df = pd.read_csv(file_path)

# ----------------------------------------------------------
# Basic checks
# ----------------------------------------------------------

required_columns = [
    "year",
    "exporter_code",
    "exporter",
    "importer_code",
    "importer",
    "product_code",
    "product",
    "trade_value",
    "quantity",
    "HS2"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing columns: {missing_columns}"
    )

# Numeric conversion
df["year"] = pd.to_numeric(
    df["year"],
    errors="coerce"
)

df["trade_value"] = pd.to_numeric(
    df["trade_value"],
    errors="coerce"
)

df["quantity"] = pd.to_numeric(
    df["quantity"],
    errors="coerce"
)

# Remove invalid observations
df = df.dropna(
    subset=[
        "year",
        "exporter",
        "importer",
        "trade_value"
    ]
).copy()

df = df[
    df["trade_value"] >= 0
].copy()

df["year"] = df["year"].astype(int)

df = df.sort_values(
    [
        "year",
        "exporter",
        "importer"
    ]
).reset_index(drop=True)

print("=" * 70)
print("Cleaned Dataset Loaded Successfully")
print("=" * 70)

print(f"Observations : {len(df):,}")
print(
    f"Period       : "
    f"{df['year'].min()} - {df['year'].max()}"
)

print(
    f"Countries    : "
    f"{len(set(df['exporter']) | set(df['importer']))}"
)

print("\nDataset preview:")

display(df.head())

Cleaned Dataset Loaded Successfully
Observations : 1,218,335
Period       : 1995 - 2024
Countries    : 233

Dataset preview:


,year,exporter_code,exporter,importer_code,importer,product_code,product,trade_value,quantity,HS2
0,1995,4,Afghanistan,208,Denmark,854211,"Electronic circuits: monolithic integrated, di...",2.758,0.002,85
1,1995,4,Afghanistan,246,Finland,854280,Electronic circuits and microassemblies: integ...,9.207,0.207,85
2,1995,8,Albania,12,Algeria,854130,"Electrical apparatus: thyristors, diacs and tr...",2.708,0.033,85
3,1995,8,Albania,380,Italy,854211,"Electronic circuits: monolithic integrated, di...",2.039,0.003,85
4,1995,8,Albania,484,Mexico,854211,"Electronic circuits: monolithic integrated, di...",1.000,0.002,85


In [3]:
# ==========================================================
# Cell 2.1 : 2024 Trade Statistics 
# ==========================================================

print("=" * 80)
print("2024 INTERNATIONAL SEMICONDUCTOR TRADE STATISTICS")
print("=" * 80)

# ----------------------------------------------------------
# Aggregate bilateral trade flows (all HS products)
# ----------------------------------------------------------

trade2024 = (

    df[df["year"] == 2024]

    .groupby(
        ["exporter", "importer"],
        as_index=False
    )

    .agg(
        total_trade_value=("trade_value", "sum")
    )

)

# Convert to Billion USD
trade2024["Trade 2024 (Billion USD)"] = (
    trade2024["total_trade_value"] / 1_000_000
)

trade2024 = trade2024.drop(
    columns="total_trade_value"
)

# ----------------------------------------------------------
# Top 25 Bilateral Trade Flows
# ----------------------------------------------------------

print("\n")
print("=" * 80)
print("Top 25 Bilateral Trade Flows (2024)")
print("=" * 80)

top25_bilateral = (

    trade2024

    .sort_values(
        "Trade 2024 (Billion USD)",
        ascending=False
    )

    .head(25)

)

display(top25_bilateral)

# ----------------------------------------------------------
# Top 25 Exporting Countries
# ----------------------------------------------------------

print("\n")
print("=" * 80)
print("Top 25 Exporting Countries (2024)")
print("=" * 80)

top25_exporters = (

    trade2024

    .groupby(
        "exporter",
        as_index=False
    )

    .agg(
        **{
            "Exports 2024 (Billion USD)":
            ("Trade 2024 (Billion USD)", "sum")
        }
    )

    .sort_values(
        "Exports 2024 (Billion USD)",
        ascending=False
    )

    .head(25)

)

display(top25_exporters)

# ----------------------------------------------------------
# Top 25 Importing Countries
# ----------------------------------------------------------

print("\n")
print("=" * 80)
print("Top 25 Importing Countries (2024)")
print("=" * 80)

top25_importers = (

    trade2024

    .groupby(
        "importer",
        as_index=False
    )

    .agg(
        **{
            "Imports 2024 (Billion USD)":
            ("Trade 2024 (Billion USD)", "sum")
        }
    )

    .sort_values(
        "Imports 2024 (Billion USD)",
        ascending=False
    )

    .head(25)

)

display(top25_importers)

2024 INTERNATIONAL SEMICONDUCTOR TRADE STATISTICS


Top 25 Bilateral Trade Flows (2024)


,exporter,importer,Trade 2024 (Billion USD)
1600,China,"China, Hong Kong SAR",83.604321
7749,Rep. of Korea,China,64.410393
6980,"Other Asia, nes",China,50.592755
6981,"Other Asia, nes","China, Hong Kong SAR",47.770213
7071,"Other Asia, nes",Singapore,45.919220
8440,Singapore,"China, Hong Kong SAR",36.751123
7826,Rep. of Korea,"Other Asia, nes",28.970702
7750,Rep. of Korea,"China, Hong Kong SAR",22.764870
7063,"Other Asia, nes",Rep. of Korea,22.583532
1710,China,Rep. of Korea,21.009840




Top 25 Exporting Countries (2024)


,exporter,Exports 2024 (Billion USD)
151,"Other Asia, nes",241.744220
43,China,228.875689
163,Rep. of Korea,172.273841
122,Malaysia,94.301007
183,Singapore,75.894634
105,Japan,62.102270
210,USA,46.925648
220,Viet Nam,40.020509
158,Philippines,26.498281
82,Germany,23.595538




Top 25 Importing Countries (2024)


,importer,Imports 2024 (Billion USD)
44,"China, Hong Kong SAR",238.927796
43,China,214.838968
183,Singapore,99.152279
152,"Other Asia, nes",78.766671
220,Viet Nam,66.028343
163,Rep. of Korea,64.363005
210,USA,62.277182
123,Malaysia,52.667407
98,India,28.984597
130,Mexico,28.167986


## 3: Graph Preprocessing


In [4]:
# ==========================================================
# Cell 3 : Graph Preprocessing
# ==========================================================

print("=" * 70)
print("Graph Preprocessing")
print("=" * 70)

# ----------------------------------------------------------
# Aggregate product-level observations into bilateral flows
# ----------------------------------------------------------

edge_features = (
    df
    .groupby(
        [
            "year",
            "exporter",
            "importer"
        ],
        as_index=False
    )
    .agg(
        total_trade_value=(
            "trade_value",
            "sum"
        ),

        total_quantity=(
            "quantity",
            "sum"
        ),

        traded_products=(
            "product_code",
            "nunique"
        )
    )
)

# ----------------------------------------------------------
# Log-transformed target
# ----------------------------------------------------------

edge_features["log_trade_value"] = np.log1p(
    edge_features["total_trade_value"]
)

# ----------------------------------------------------------
# Global country mapping
# Same node ID across all years
# ----------------------------------------------------------

countries = sorted(
    set(edge_features["exporter"])
    |
    set(edge_features["importer"])
)

country_to_id = {
    country: idx
    for idx, country in enumerate(countries)
}

id_to_country = {
    idx: country
    for country, idx in country_to_id.items()
}

edge_features["source"] = (
    edge_features["exporter"]
    .map(country_to_id)
)

edge_features["target"] = (
    edge_features["importer"]
    .map(country_to_id)
)

print(
    f"Number of countries       : {len(countries)}"
)

print(
    f"Number of bilateral flows : {len(edge_features):,}"
)

print(
    f"Period                    : "
    f"{edge_features['year'].min()} - "
    f"{edge_features['year'].max()}"
)

display(edge_features.head())

Graph Preprocessing
Number of countries       : 233
Number of bilateral flows : 272,196
Period                    : 1995 - 2024


,year,exporter,importer,total_trade_value,total_quantity,traded_products,log_trade_value,source,target
0,1995,Afghanistan,Denmark,2.758,0.002,1,1.323887,0,62
1,1995,Afghanistan,Finland,9.207,0.207,1,2.323074,0,77
2,1995,Albania,Algeria,2.708,0.033,1,1.310493,1,2
3,1995,Albania,Italy,2.039,0.003,1,1.111529,1,105
4,1995,Albania,Mexico,1.000,0.002,1,0.693147,1,132


## 4 : Construction of Yearly Graphs



In [5]:
# ==========================================================
# Cell 4 : Construction of Yearly Graphs
# ==========================================================

print("=" * 70)
print("Construction of Yearly Trade Graphs")
print("=" * 70)

yearly_graph_data = {}

years = sorted(
    edge_features["year"].unique()
)

for year in years:

    yearly = edge_features[
        edge_features["year"] == year
    ].copy()

    edge_index = torch.tensor(
        [
            yearly["source"].values,
            yearly["target"].values
        ],
        dtype=torch.long
    )

    yearly_graph_data[year] = {
        "edge_index": edge_index,
        "dataframe": yearly
    }

    print(
        f"{year} : "
        f"{len(countries)} nodes | "
        f"{edge_index.shape[1]} directed edges"
    )

Construction of Yearly Trade Graphs
1995 : 233 nodes | 4315 directed edges
1996 : 233 nodes | 4699 directed edges
1997 : 233 nodes | 5034 directed edges
1998 : 233 nodes | 5311 directed edges
1999 : 233 nodes | 5736 directed edges
2000 : 233 nodes | 6968 directed edges
2001 : 233 nodes | 7314 directed edges
2002 : 233 nodes | 7476 directed edges
2003 : 233 nodes | 7866 directed edges
2004 : 233 nodes | 8308 directed edges
2005 : 233 nodes | 8477 directed edges
2006 : 233 nodes | 8996 directed edges
2007 : 233 nodes | 9279 directed edges
2008 : 233 nodes | 9500 directed edges
2009 : 233 nodes | 9697 directed edges
2010 : 233 nodes | 9994 directed edges
2011 : 233 nodes | 10132 directed edges
2012 : 233 nodes | 10398 directed edges
2013 : 233 nodes | 10518 directed edges
2014 : 233 nodes | 10551 directed edges
2015 : 233 nodes | 10884 directed edges
2016 : 233 nodes | 11090 directed edges
2017 : 233 nodes | 11179 directed edges
2018 : 233 nodes | 11137 directed edges
2019 : 233 nodes | 1

## 5 : Node Feature Engineering

In [6]:
# ==========================================================
# Cell 5 : Node Feature Engineering
# ==========================================================

print("=" * 70)
print("Node Feature Engineering")
print("=" * 70)

node_features_by_year = {}

NUM_NODES = len(countries)

for year in years:

    yearly = edge_features[
        edge_features["year"] == year
    ]

    features = np.zeros(
        (NUM_NODES, 6),
        dtype=np.float32
    )

    # ------------------------------------------------------
    # Degree features
    # ------------------------------------------------------

    out_degree = (
        yearly
        .groupby("source")["target"]
        .nunique()
    )

    in_degree = (
        yearly
        .groupby("target")["source"]
        .nunique()
    )

    # ------------------------------------------------------
    # Trade aggregates
    # ------------------------------------------------------

    exports = (
        yearly
        .groupby("source")["total_trade_value"]
        .sum()
    )

    imports = (
        yearly
        .groupby("target")["total_trade_value"]
        .sum()
    )

    for node in range(NUM_NODES):

        out_d = out_degree.get(node, 0)

        in_d = in_degree.get(node, 0)

        export_value = exports.get(node, 0)

        import_value = imports.get(node, 0)

        total_trade = (
            export_value
            +
            import_value
        )

        features[node] = [

            out_d,

            in_d,

            out_d + in_d,

            np.log1p(export_value),

            np.log1p(import_value),

            np.log1p(total_trade)

        ]

    node_features_by_year[year] = features

print(
    "Node features created successfully."
)

print(
    "Features:"
)

print(
    [
        "Out Degree",
        "In Degree",
        "Total Degree",
        "Log Exports",
        "Log Imports",
        "Log Total Trade"
    ]
)

Node Feature Engineering
Node features created successfully.
Features:
['Out Degree', 'In Degree', 'Total Degree', 'Log Exports', 'Log Imports', 'Log Total Trade']


## 6 : Edge Feature Engineering

In [7]:
# ==========================================================
# Cell 6 : Edge Feature Engineering
# ==========================================================

print("=" * 70)
print("Edge Feature Engineering")
print("=" * 70)

prediction_samples = {}

for year in years:

    previous_year = year - 1

    if previous_year not in years:
        continue

    current = edge_features[
        edge_features["year"] == year
    ].copy()

    previous = edge_features[
        edge_features["year"] == previous_year
    ][
        [
            "source",
            "target",
            "total_trade_value"
        ]
    ].copy()

    previous = previous.rename(
        columns={
            "total_trade_value":
            "previous_trade_value"
        }
    )

    current = current.merge(
        previous,
        on=["source", "target"],
        how="left"
    )

    current[
        "previous_trade_value"
    ] = current[
        "previous_trade_value"
    ].fillna(0)

    current[
        "log_previous_trade"
    ] = np.log1p(
        current["previous_trade_value"]
    )

    prediction_samples[year] = current

print(
    f"Prediction years : "
    f"{min(prediction_samples.keys())} - "
    f"{max(prediction_samples.keys())}"
)

display(
    prediction_samples[
        min(prediction_samples.keys())
    ].head()
)

Edge Feature Engineering
Prediction years : 1996 - 2024


,year,exporter,importer,total_trade_value,total_quantity,traded_products,log_trade_value,source,target,previous_trade_value,log_previous_trade
0,1996,Afghanistan,Finland,1.477,0.062,1,0.907048,0,77,9.207,2.323074
1,1996,Afghanistan,Slovenia,0.819,0.009,1,0.598287,0,190,0.000,0.000000
2,1996,Afghanistan,Southern African Customs Union (...1999),3.316,0.001,1,1.462329,0,195,0.000,0.000000
3,1996,Afghanistan,Switzerland,43.222,0.007,1,3.789222,0,203,0.000,0.000000
4,1996,Albania,Brazil,2.012,0.003,1,1.102604,1,31,0.000,0.000000


## 7 : PyTorch Geometric Conversion



In [8]:
# ==========================================================
# Cell 7 : PyTorch Geometric Conversion
# ==========================================================

print("=" * 70)
print("PyTorch Geometric Conversion")
print("=" * 70)

pyg_graphs = {}

for target_year in sorted(
    prediction_samples.keys()
):

    input_year = target_year - 1

    # ------------------------------------------------------
    # Node features from t-1
    # ------------------------------------------------------

    x = torch.tensor(
        node_features_by_year[input_year],
        dtype=torch.float
    )

    # ------------------------------------------------------
    # Graph structure from t-1
    # ------------------------------------------------------

    message_edge_index = (
        yearly_graph_data[input_year]
        ["edge_index"]
        .clone()
    )

    # ------------------------------------------------------
    # Prediction edges from year t
    # ------------------------------------------------------

    samples = prediction_samples[
        target_year
    ]

    target_edge_index = torch.tensor(
        [
            samples["source"].values,
            samples["target"].values
        ],
        dtype=torch.long
    )

    # Target = log(1 + trade value)
    y = torch.tensor(
        samples["log_trade_value"].values,
        dtype=torch.float
    )

    # Lagged bilateral flow
    lag_trade = torch.tensor(
        samples[
            "log_previous_trade"
        ].values,
        dtype=torch.float
    ).view(-1, 1)

    pyg_graphs[target_year] = Data(

        x=x,

        edge_index=message_edge_index,

        target_edge_index=target_edge_index,

        y=y,

        lag_trade=lag_trade

    )

print(
    f"{len(pyg_graphs)} temporal PyG graphs created."
)

example_year = sorted(
    pyg_graphs.keys()
)[0]

print(
    f"\nExample ({example_year}):"
)

print(
    pyg_graphs[example_year]
)

PyTorch Geometric Conversion
29 temporal PyG graphs created.

Example (1996):
Data(x=[233, 6], edge_index=[2, 4315], y=[4699], target_edge_index=[2, 4699], lag_trade=[4699, 1])


## 8 : Training 



In [9]:
# ==========================================================
# Cell 8 : Full Training Dataset Preparation
# ==========================================================

print("=" * 70)
print("Full Training Dataset Preparation")
print("=" * 70)

# ----------------------------------------------------------
# Use all available years for model training
# ----------------------------------------------------------

training_years = sorted(
    pyg_graphs.keys()
)

forecast_year = 2025

print(
    f"Training period : "
    f"{min(training_years)} - {max(training_years)}"
)

print(
    f"Forecast year   : "
    f"{forecast_year}"
)

# ----------------------------------------------------------
# Standardize node features
# Fit using all available training data
# ----------------------------------------------------------

scaler = StandardScaler()

training_features = np.vstack(
    [
        node_features_by_year[y - 1]
        for y in training_years
    ]
)

scaler.fit(
    training_features
)

# ----------------------------------------------------------
# Apply feature standardization
# ----------------------------------------------------------

for target_year in training_years:

    input_year = target_year - 1

    scaled_features = scaler.transform(
        node_features_by_year[input_year]
    )

    pyg_graphs[target_year].x = torch.tensor(
        scaled_features,
        dtype=torch.float
    )

print(
    "\nNode features standardized using all available training data."
)

print(
    "Dataset is ready for final model training."
)

Full Training Dataset Preparation
Training period : 1996 - 2024
Forecast year   : 2025

Node features standardized using all available training data.
Dataset is ready for final model training.


## 9 : GraphSAGE Architecture



### TradeGraph Architecture

TradeGraph is a hybrid multi-branch Graph Neural Network (GNN) designed for international trade flow prediction. Instead of relying on a single graph convolution operator, the proposed architecture assumes that different message-passing mechanisms capture complementary aspects of international trade networks. Consequently, the same input graph is processed simultaneously by three independent GNN branches before their learned representations are adaptively fused into a unified embedding for trade flow prediction.

The first branch is based on **Graph Convolutional Networks (GCN)**, whose objective is to capture the **global structural organization** of the trade network. By recursively propagating information across multiple neighborhood levels, GCN learns node representations that reflect indirect relationships and long-range dependencies between countries. This enables the model to account for global supply chain effects, where trade interactions between distant economies may still influence bilateral trade flows.

The second branch employs **GraphSAGE**, which focuses on **local neighborhood aggregation**. Unlike GCN, GraphSAGE explicitly learns an aggregation function over the immediate neighbors of each node, allowing the model to capture direct bilateral trade relationships more effectively while maintaining scalability to large graphs. This branch is particularly suitable for modeling international trade because most trade flows are primarily influenced by a country's closest commercial partners.

The third branch relies on **Graph Attention Networks (GAT)** to learn **attention-weighted neighborhood representations**. Instead of treating all neighboring countries equally, GAT automatically assigns different importance scores to each trading partner according to its contribution to the prediction task. As a result, strategically important countries within the global semiconductor supply chain receive greater influence during message passing, allowing the model to emphasize the most informative trade relationships.

The embeddings generated by the GCN, GraphSAGE, and GAT branches are then concatenated and passed to a **Gated Fusion Network**. This module constitutes the core innovation of TradeGraph, as it learns adaptive fusion weights that dynamically determine the relative contribution of each branch. Rather than assuming that one aggregation mechanism is universally superior, the gating network allows the model to decide whether global structural information, local interactions, or attention-based representations should receive greater emphasis for a given trade context.

The weighted embedding is subsequently refined by a **Fusion Network**, composed of fully connected layers with ReLU activation and dropout regularization. This refinement stage transforms the fused representation into a more expressive latent space while reducing redundancy among the three branches.

To improve optimization stability and preserve the information learned by the individual branches, TradeGraph incorporates a **Residual Connection**. Specifically, the refined fused embedding is combined with the average of the three branch embeddings, facilitating gradient propagation during training and preventing the loss of useful structural information.

Finally, the resulting node embeddings are passed to a **Directed Edge Regression Decoder**. For each directed trade relationship, the decoder concatenates the embedding of the exporting country, the embedding of the importing country, and the lagged trade value from the previous year. This edge representation is processed by a multilayer perceptron (MLP) to estimate the future bilateral trade flow. By integrating global topology, local neighborhood information, attention-based partner importance, adaptive fusion, and temporal trade dynamics, TradeGraph provides a unified framework for predicting international trade flows.

In [10]:
# ==========================================================
# Part 1 : Initialization
# ==========================================================

class TradeGraphPredictor(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dim,
        embedding_dim,
        num_layers,
        heads,
        dropout
    ):

        super().__init__()

        self.dropout = dropout

        self.embedding_dim = embedding_dim

        ####################################################
        # Branch 1 : GCN
        ####################################################

        self.gcn_convs = nn.ModuleList()

        self.gcn_convs.append(

            GCNConv(

                input_dim,

                hidden_dim

            )

        )

        current_dim = hidden_dim

        for _ in range(max(0, num_layers - 2)):

            self.gcn_convs.append(

                GCNConv(

                    current_dim,

                    hidden_dim

                )

            )

        if num_layers > 1:

            self.gcn_convs.append(

                GCNConv(

                    hidden_dim,

                    embedding_dim

                )

            )

        else:

            self.gcn_convs[0] = GCNConv(

                input_dim,

                embedding_dim

            )

        ####################################################
        # Branch 2 : GraphSAGE
        ####################################################

        self.sage_convs = nn.ModuleList()

        self.sage_convs.append(

            SAGEConv(

                input_dim,

                hidden_dim

            )

        )

        current_dim = hidden_dim

        for _ in range(max(0, num_layers - 2)):

            self.sage_convs.append(

                SAGEConv(

                    current_dim,

                    hidden_dim

                )

            )

        if num_layers > 1:

            self.sage_convs.append(

                SAGEConv(

                    hidden_dim,

                    embedding_dim

                )

            )

        else:

            self.sage_convs[0] = SAGEConv(

                input_dim,

                embedding_dim

            )

        ####################################################
        # Branch 3 : GAT
        ####################################################

        self.gat_convs = nn.ModuleList()

        self.gat_convs.append(

            GATConv(

                input_dim,

                hidden_dim,

                heads=heads,

                concat=True,

                dropout=dropout

            )

        )

        current_dim = hidden_dim * heads

        for _ in range(max(0, num_layers - 2)):

            self.gat_convs.append(

                GATConv(

                    current_dim,

                    hidden_dim,

                    heads=heads,

                    concat=True,

                    dropout=dropout

                )

            )

            current_dim = hidden_dim * heads

        if num_layers > 1:

            self.gat_convs.append(

                GATConv(

                    current_dim,

                    embedding_dim,

                    heads=1,

                    concat=False,

                    dropout=dropout

                )

            )

        else:

            self.gat_convs[0] = GATConv(

                input_dim,

                embedding_dim,

                heads=1,

                concat=False,

                dropout=dropout

            )

        ####################################################
        # Gated Fusion Network
        ####################################################

        # Learn the importance of each branch

        self.gate_network = nn.Sequential(

            nn.Linear(

                embedding_dim * 3,

                embedding_dim

            ),

            nn.ReLU(),

            nn.Dropout(

                dropout

            ),

            nn.Linear(

                embedding_dim,

                3

            ),

            nn.Softmax(dim=1)

        )

        ####################################################
        # Fusion refinement
        ####################################################

        self.fusion_network = nn.Sequential(

            nn.Linear(

                embedding_dim,

                hidden_dim

            ),

            nn.ReLU(),

            nn.Dropout(

                dropout

            ),

            nn.Linear(

                hidden_dim,

                embedding_dim

            )

        )

        ####################################################
        # Directed Edge Regression Decoder
        ####################################################

        self.edge_decoder = nn.Sequential(

            nn.Linear(

                embedding_dim * 2 + 1,

                hidden_dim

            ),

            nn.ReLU(),

            nn.Dropout(

                dropout

            ),

            nn.Linear(

                hidden_dim,

                hidden_dim // 2

            ),

            nn.ReLU(),

            nn.Linear(

                hidden_dim // 2,

                1

            )

        )

    ####################################################
    # Part 2 : Branch 1 - GCN Encoder
    ####################################################

    def encode_gcn(
        self,
        x,
        edge_index
    ):

        for i, conv in enumerate(self.gcn_convs):

            x = conv(
                x,
                edge_index
            )

            if i != len(self.gcn_convs) - 1:

                x = F.relu(x)

                x = F.dropout(
                    x,
                    p=self.dropout,
                    training=self.training
                )

        return x


    ####################################################
    # Branch 2 : GraphSAGE Encoder
    ####################################################

    def encode_sage(
        self,
        x,
        edge_index
    ):

        for i, conv in enumerate(self.sage_convs):

            x = conv(
                x,
                edge_index
            )

            if i != len(self.sage_convs) - 1:

                x = F.relu(x)

                x = F.dropout(
                    x,
                    p=self.dropout,
                    training=self.training
                )

        return x


    ####################################################
    # Branch 3 : GAT Encoder
    ####################################################

    def encode_gat(
        self,
        x,
        edge_index
    ):

        for i, conv in enumerate(self.gat_convs):

            x = conv(
                x,
                edge_index
            )

            if i != len(self.gat_convs) - 1:

                x = F.elu(x)

                x = F.dropout(
                    x,
                    p=self.dropout,
                    training=self.training
                )

        return x
        
    ####################################################
    # Part 3 : fuse() + decode() : Gated Fusion
    ####################################################

    def fuse(
        self,
        z_gcn,
        z_sage,
        z_gat
    ):

        # -----------------------------------------------
        # Concatenate the three representations
        # -----------------------------------------------

        z_concat = torch.cat(

            [

                z_gcn,

                z_sage,

                z_gat

            ],

            dim=1

        )

        # -----------------------------------------------
        # Learn branch importance
        # -----------------------------------------------

        gate_weights = self.gate_network(

            z_concat

        )

        alpha = gate_weights[:, 0].unsqueeze(1)

        beta = gate_weights[:, 1].unsqueeze(1)

        gamma = gate_weights[:, 2].unsqueeze(1)

        # -----------------------------------------------
        # Weighted fusion
        # -----------------------------------------------

        z_fused = (

            alpha * z_gcn +

            beta * z_sage +

            gamma * z_gat

        )

        # -----------------------------------------------
        # Fusion refinement
        # -----------------------------------------------

        z_final = self.fusion_network(

            z_fused

        )

        return z_final


    ####################################################
    # Directed Edge Regression Decoder
    ####################################################

    def decode(
        self,
        z,
        target_edge_index,
        lag_trade
    ):

        source = target_edge_index[0]

        target = target_edge_index[1]

        edge_representation = torch.cat(

            [

                z[source],

                z[target],

                lag_trade

            ],

            dim=1

        )

        prediction = self.edge_decoder(

            edge_representation

        )

        return prediction.squeeze(-1)
        
    ####################################################
    # Part 4 : Forward
    ####################################################

    def forward(
        self,
        data
    ):

        # ------------------------------------------------
        # Branch 1 : Global Structural Representation
        # (GCN)
        # ------------------------------------------------

        z_gcn = self.encode_gcn(

            data.x,

            data.edge_index

        )

        # ------------------------------------------------
        # Branch 2 : Local Neighborhood Representation
        # (GraphSAGE)
        # ------------------------------------------------

        z_sage = self.encode_sage(

            data.x,

            data.edge_index

        )

        # ------------------------------------------------
        # Branch 3 : Partner Importance Representation
        # (GAT)
        # ------------------------------------------------

        z_gat = self.encode_gat(

            data.x,

            data.edge_index

        )

        # ------------------------------------------------
        # Gated Fusion
        # ------------------------------------------------

        z = self.fuse(

            z_gcn,

            z_sage,

            z_gat

        )

        # ------------------------------------------------
        # Residual Connection
        # Improves optimization stability
        # ------------------------------------------------

        z = z + (

            z_gcn +

            z_sage +

            z_gat

        ) / 3.0

        # ------------------------------------------------
        # Directed Edge Regression
        # ------------------------------------------------

        prediction = self.decode(

            z,

            data.target_edge_index,

            data.lag_trade

        )

        return prediction


# ==========================================================
# TradeGraph Architecture Successfully Defined
# ==========================================================

print(
    "TradeGraphPredictor architecture defined successfully."
)

print(
    "Branches : GCN + GraphSAGE + GAT"
)

print(
    "Fusion : Gated Attention Fusion (Softmax)"
)

print(
    "Decoder : Directed Edge Regression Decoder"
)

print(
    "Output : Trade Flow Prediction"
)
    

TradeGraphPredictor architecture defined successfully.
Branches : GCN + GraphSAGE + GAT
Fusion : Gated Attention Fusion (Softmax)
Decoder : Directed Edge Regression Decoder
Output : Trade Flow Prediction


## 10 : Hyperparameter Optimization with Optuna

In [11]:
# ==========================================================
# Cell 10 : Best Hyperparameters (already found by optuna)
# ==========================================================

print("=" * 70)
print("Loading Best TradeGraph Hyperparameters")
print("=" * 70)

best_params = {

    "hidden_dim": 256,

    "embedding_dim": 32,

    "num_layers": 2,

    "heads": 8,

    "dropout": 0.020026631197368593,

    "learning_rate": 0.0015639443017267521,

    "weight_decay": 6.9049606227594705e-06

}

best_epoch = 141

print("Best hyperparameters loaded successfully.")

for parameter, value in best_params.items():

    print(f"{parameter:<20}: {value}")

print("\nReady for final model training.")

Loading Best TradeGraph Hyperparameters
Best hyperparameters loaded successfully.
hidden_dim          : 256
embedding_dim       : 32
num_layers          : 2
heads               : 8
dropout             : 0.020026631197368593
learning_rate       : 0.0015639443017267521
weight_decay        : 6.9049606227594705e-06

Ready for final model training.


In [12]:
# ==========================================================
# Cell 10B : Training Function
# ==========================================================

print("=" * 70)
print("Training Function")
print("=" * 70)


# ==========================================================
# Training Function
# ==========================================================

def train_one_epoch(
    model,
    optimizer,
    years_subset
):

    model.train()

    total_loss = 0.0

    for year in years_subset:

        data = pyg_graphs[year].to(device)

        optimizer.zero_grad()

        predictions = model(data)

        loss = F.mse_loss(
            predictions,
            data.y
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(years_subset)


print(
    "Training function successfully defined."
)

Training Function
Training function successfully defined.


## 11 : Final Model Training

In [13]:
# ==========================================================
# Cell 11 : Final TradeGraph Model Training 
# ==========================================================

print("=" * 70)
print("Final TradeGraph Model Training")
print("=" * 70)

# ----------------------------------------------------------
# Reset random seeds before final model training
# ----------------------------------------------------------

random.seed(SEED)

np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():

    torch.cuda.manual_seed(SEED)

    torch.cuda.manual_seed_all(SEED)

    torch.backends.cudnn.deterministic = True

    torch.backends.cudnn.benchmark = False


# ----------------------------------------------------------
# Initialize Best TradeGraph Model
# ----------------------------------------------------------

best_model = TradeGraphPredictor(

    input_dim=
        pyg_graphs[
            training_years[0]
        ].x.shape[1],

    hidden_dim=
        best_params["hidden_dim"],

    embedding_dim=
        best_params["embedding_dim"],

    num_layers=
        best_params["num_layers"],

    heads=
        best_params["heads"],

    dropout=
        best_params["dropout"]

).to(device)


# ----------------------------------------------------------
# Optimizer
# ----------------------------------------------------------

optimizer = torch.optim.Adam(

    best_model.parameters(),

    lr=
        best_params["learning_rate"],

    weight_decay=
        best_params["weight_decay"]

)


# ----------------------------------------------------------
# Training Configuration
# ----------------------------------------------------------

MAX_EPOCHS = 141     

train_rmse_history = []


# ----------------------------------------------------------
# Final Training Loop
# ----------------------------------------------------------

for epoch in range(

    1,

    MAX_EPOCHS + 1

):

    train_mse = train_one_epoch(

        best_model,

        optimizer,

        training_years

    )

    train_rmse = np.sqrt(

        train_mse

    )

    train_rmse_history.append(

        train_rmse

    )

    if epoch % 25 == 0 or epoch == 1 or epoch == MAX_EPOCHS:

        print(

            f"Epoch {epoch:4d} | "

            f"Train RMSE: "

            f"{train_rmse:.6f}"

        )


# ----------------------------------------------------------
# Save Final Model
# ----------------------------------------------------------

torch.save(

    best_model.state_dict(),

    "best_tradegraph_model.pth"

)


print("\n")

print("=" * 70)

print("TradeGraph Training Completed")

print("=" * 70)

print(

    f"Training epochs : {MAX_EPOCHS}"

)

print(

    f"Final Train RMSE : {train_rmse_history[-1]:.6f}"

)


print(

    "\nFinal model saved as : "

    "best_tradegraph_model.pth"

)


# ----------------------------------------------------------
# Training Summary
# ----------------------------------------------------------

print("\n")

print("=" * 70)

print("TradeGraph Final Model Ready")

print("=" * 70)

print(

    f"The optimized TradeGraph model has been retrained on "

    f"all available years ({min(training_years)}–{max(training_years)}) "

    f"and is ready for {forecast_year} forecasting."

)

Final TradeGraph Model Training


[W729 15:33:15.384735634 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 94371840 bytes (free: 55181312, total: 21009465344).


Epoch    1 | Train RMSE: 2.773534
Epoch   25 | Train RMSE: 1.384211
Epoch   50 | Train RMSE: 1.379484
Epoch   75 | Train RMSE: 1.370098
Epoch  100 | Train RMSE: 1.365677
Epoch  125 | Train RMSE: 1.369478
Epoch  141 | Train RMSE: 1.362720


TradeGraph Training Completed
Training epochs : 141
Final Train RMSE : 1.362720

Final model saved as : best_tradegraph_model.pth


TradeGraph Final Model Ready
The optimized TradeGraph model has been retrained on all available years (1996–2024) and is ready for 2025 forecasting.


In [14]:
# ==========================================================
# Cell 12A : Prepare Forecast Graph for 2025
# ==========================================================

from torch_geometric.data import Data

print("=" * 70)
print("Preparing Forecast Graph for 2025")
print("=" * 70)

# ----------------------------------------------------------
# Latest available graph
# ----------------------------------------------------------

latest_year = max(pyg_graphs.keys())

forecast_base_year = latest_year

latest_graph = pyg_graphs[forecast_base_year]

# ----------------------------------------------------------
# Build forecast graph
# ----------------------------------------------------------

forecast_graph = Data(

    x=latest_graph.x.clone(),

    edge_index=latest_graph.edge_index.clone(),

    target_edge_index=latest_graph.target_edge_index.clone(),

    lag_trade=latest_graph.lag_trade.clone()

)

print(f"Latest available year : {forecast_base_year}")
print(f"Forecast year         : {forecast_year}")
print()
print(f"Number of countries   : {forecast_graph.num_nodes}")
print(f"Prediction edges      : {forecast_graph.target_edge_index.shape[1]}")
print("Target values (y)     : Not available (forecast mode)")
print()
print("Forecast graph successfully created.")

Preparing Forecast Graph for 2025
Latest available year : 2024
Forecast year         : 2025

Number of countries   : 233
Prediction edges      : 10835
Target values (y)     : Not available (forecast mode)

Forecast graph successfully created.


## 12 : Prediction

In [15]:
# ==========================================================
# Cell 12 : Forecast Trade Flows for 2025
# ==========================================================

print("=" * 70)
print(f"TradeGraph Forecast for {forecast_year}")
print("=" * 70)

# ----------------------------------------------------------
# Evaluation mode
# ----------------------------------------------------------

best_model.eval()

with torch.no_grad():

    forecast_graph = forecast_graph.to(device)

    pred_log = best_model(forecast_graph)

    pred_log = pred_log.cpu().numpy()


# ----------------------------------------------------------
# Back-transform predictions
# ----------------------------------------------------------

pred_trade = np.expm1(pred_log)

pred_trade = np.maximum(pred_trade, 0)

# ----------------------------------------------------------
# Retrieve 2024 trade flows
# ----------------------------------------------------------

forecast_samples = prediction_samples[
    forecast_base_year
].copy().reset_index(drop=True)

# ----------------------------------------------------------
# Build Forecast Output Table
# ----------------------------------------------------------

# Start from the official observed 2024 trade flows
forecast_output = trade2024.rename(
    columns={
        "exporter": "Exporter",
        "importer": "Importer"
    }
).copy()

# Add forecast values
forecast_output["Forecast 2025 (USD)"] = pred_trade

forecast_output["Forecast 2025 (Billion USD)"] = (
    forecast_output["Forecast 2025 (USD)"] / 1e6
)

# Recover USD values only for growth calculation
forecast_output["Trade 2024 (USD)"] = (
    forecast_output["Trade 2024 (Billion USD)"] * 1e6
)

# ----------------------------------------------------------
# Growth Rate
# ----------------------------------------------------------

forecast_output["Growth (%)"] = np.where(

    forecast_output["Trade 2024 (USD)"] > 0,

    (
        (
            forecast_output["Forecast 2025 (USD)"]
            -
            forecast_output["Trade 2024 (USD)"]
        )
        /
        forecast_output["Trade 2024 (USD)"]
    ) * 100,

    np.nan

)


# ----------------------------------------------------------
# Keep useful columns
# ----------------------------------------------------------

forecast_output = forecast_output[
    [
        "Exporter",
        "Importer",
        "Trade 2024 (Billion USD)",
        "Forecast 2025 (Billion USD)",
        "Growth (%)"
    ]
]

# ----------------------------------------------------------
# Round numerical values
# ----------------------------------------------------------

forecast_output = forecast_output.round({
    "Trade 2024 (Billion USD)": 3,
    "Forecast 2025 (Billion USD)": 3,
    "Growth (%)": 3
})

# Display + before positive growth
forecast_output["Growth (%)"] = forecast_output["Growth (%)"].map(
    lambda x: f"{x:+.3f}" if pd.notna(x) else ""
)

# ----------------------------------------------------------
# Summary
# ----------------------------------------------------------

print()

print("=" * 70)
print("Forecast Summary")
print("=" * 70)

print(
    f"Number of trade flows : {len(forecast_output):,}"
)

print(
    f"Forecast year         : {forecast_year}"
)

print(
    f"Predicted total trade : "
    f"{forecast_output['Forecast 2025 (Billion USD)'].sum():,.2f} Billion USD"
)

display(
    forecast_output.head(10)
)

# ----------------------------------------------------------
# Top 25 Largest Forecast Trade Flows
# ----------------------------------------------------------
pd.set_option("display.float_format", "{:.3f}".format)
print()

print("=" * 70)
print("Top 25 Largest Forecast Trade Flows")
print("=" * 70)

top25_forecast = (
    forecast_output
    .sort_values(
        "Forecast 2025 (Billion USD)",
        ascending=False
    )
    .head(25)
)

display(top25_forecast)

#----------------------------------------------------------
# Save Top 25 to Excel
#----------------------------------------------------------

top25_forecast.to_excel(
    "Top25_Forecast_Trade_Flows_2025.xlsx",
    index=False
)

print("\nTop 25 saved as: Top25_Forecast_Trade_Flows_2025.xlsx")


# ----------------------------------------------------------
# Top 25 Forecast Exporting Countries
# ----------------------------------------------------------

print()

print("=" * 70)
print("Top 25 Forecast Exporting Countries")
print("=" * 70)

# Real exports (2024)
exports2024 = (

    trade2024

    .groupby(
        "exporter",
        as_index=False
    )

    .agg(
        {
            "Trade 2024 (Billion USD)": "sum"
        }
    )

)

# Forecast exports (2025)
exports2025 = (

    forecast_output

    .groupby(
        "Exporter",
        as_index=False
    )

    .agg(
        {
            "Forecast 2025 (Billion USD)": "sum"
        }
    )

)

# Merge
top25_exports = (

    exports2024

    .merge(
        exports2025,
        left_on="exporter",
        right_on="Exporter",
        how="inner"
    )

    .drop(columns="Exporter")

    .rename(
        columns={
            "exporter": "Exporter",
            "Trade 2024 (Billion USD)": "Exports 2024 (Billion USD)",
            "Forecast 2025 (Billion USD)": "Forecast Exports 2025 (Billion USD)"
        }
    )

    .sort_values(
        "Forecast Exports 2025 (Billion USD)",
        ascending=False
    )

    .head(25)

)

display(top25_exports)

# ----------------------------------------------------------
# Save to Excel
# ----------------------------------------------------------

top25_exports.to_excel(
    "Top25_Forecast_Exporting_Countries_2025.xlsx",
    index=False
)

print(
    "\nTop 25 exporting countries saved as : "
    "Top25_Forecast_Exporting_Countries_2025.xlsx"
)


# ----------------------------------------------------------
# Top 25 Forecast Importing Countries
# ----------------------------------------------------------

print()

print("=" * 70)
print("Top 25 Forecast Importing Countries")
print("=" * 70)

# Real imports (2024)
imports2024 = (

    trade2024

    .groupby(
        "importer",
        as_index=False
    )

    .agg(
        {
            "Trade 2024 (Billion USD)": "sum"
        }
    )

)

# Forecast imports (2025)
imports2025 = (

    forecast_output

    .groupby(
        "Importer",
        as_index=False
    )

    .agg(
        {
            "Forecast 2025 (Billion USD)": "sum"
        }
    )

)

# Merge
top25_imports = (

    imports2024

    .merge(
        imports2025,
        left_on="importer",
        right_on="Importer",
        how="inner"
    )

    .drop(columns="Importer")

    .rename(
        columns={
            "importer": "Importer",
            "Trade 2024 (Billion USD)": "Imports 2024 (Billion USD)",
            "Forecast 2025 (Billion USD)": "Forecast Imports 2025 (Billion USD)"
        }
    )

    .sort_values(
        "Forecast Imports 2025 (Billion USD)",
        ascending=False
    )

    .head(25)

)

display(top25_imports)

# ----------------------------------------------------------
# Save to Excel
# ----------------------------------------------------------

top25_imports.to_excel(
    "Top25_Forecast_Importing_Countries_2025.xlsx",
    index=False
)

print(
    "\nTop 25 importing countries saved as : "
    "Top25_Forecast_Importing_Countries_2025.xlsx"
)

TradeGraph Forecast for 2025

Forecast Summary
Number of trade flows : 10,835
Forecast year         : 2025
Predicted total trade : 1,255.89 Billion USD


,Exporter,Importer,Trade 2024 (Billion USD),Forecast 2025 (Billion USD),Growth (%)
0,Afghanistan,Austria,0.0,0.0,+68.566
1,Afghanistan,Bolivia (Plurinational State of),0.0,0.0,+140.495
2,Afghanistan,Canada,0.0,0.0,-51.745
3,Afghanistan,Czechia,0.0,0.0,+349.557
4,Afghanistan,France,0.0,0.0,+229.846
5,Afghanistan,Germany,0.0,0.0,+15.753
6,Afghanistan,Indonesia,0.0,0.0,+10012.277
7,Afghanistan,Ireland,0.0,0.0,+100.208
8,Afghanistan,Luxembourg,0.0,0.0,+1252.431
9,Afghanistan,Malaysia,0.0,0.0,+408.058



Top 25 Largest Forecast Trade Flows


,Exporter,Importer,Trade 2024 (Billion USD),Forecast 2025 (Billion USD),Growth (%)
1600,China,"China, Hong Kong SAR",83.604,96.638,+15.590
6980,"Other Asia, nes",China,50.593,72.926,+44.143
7749,Rep. of Korea,China,64.410,68.179,+5.851
6981,"Other Asia, nes","China, Hong Kong SAR",47.770,63.938,+33.845
7071,"Other Asia, nes",Singapore,45.919,41.826,-8.913
8440,Singapore,"China, Hong Kong SAR",36.751,35.629,-3.053
5649,Malaysia,China,20.971,29.317,+39.801
4862,Japan,China,17.403,25.960,+49.169
1710,China,Rep. of Korea,21.010,24.589,+17.038
1699,China,"Other Asia, nes",19.166,24.527,+27.972



Top 25 saved as: Top25_Forecast_Trade_Flows_2025.xlsx

Top 25 Forecast Exporting Countries


,Exporter,Exports 2024 (Billion USD),Forecast Exports 2025 (Billion USD)
151,"Other Asia, nes",241.744,284.186
43,China,228.876,282.083
163,Rep. of Korea,172.274,150.448
122,Malaysia,94.301,104.496
105,Japan,62.102,74.342
183,Singapore,75.895,73.300
210,USA,46.926,53.412
220,Viet Nam,40.021,41.744
158,Philippines,26.498,38.450
82,Germany,23.596,33.402



Top 25 exporting countries saved as : Top25_Forecast_Exporting_Countries_2025.xlsx

Top 25 Forecast Importing Countries


,Importer,Imports 2024 (Billion USD),Forecast Imports 2025 (Billion USD)
43,China,214.839,272.215
44,"China, Hong Kong SAR",238.928,267.095
183,Singapore,99.152,98.148
152,"Other Asia, nes",78.767,71.655
210,USA,62.277,66.740
163,Rep. of Korea,64.363,64.472
220,Viet Nam,66.028,51.937
123,Malaysia,52.667,50.231
83,Germany,26.020,42.798
106,Japan,27.424,37.635



Top 25 importing countries saved as : Top25_Forecast_Importing_Countries_2025.xlsx
